In [12]:
!pip uninstall -y dgl torchdata

!pip install -q torch==2.2.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

# Install torchdata compatible with torch 2.2
!pip install -q torchdata==0.7.1

# Install DGL for cu121
!pip install dgl -q -f https://data.dgl.ai/wheels/torch-2.2/cu121/repo.html

# Install DGL LifeSci
!pip install -q dgllife rdkit pandas

Found existing installation: dgl 2.4.0+cu121
Uninstalling dgl-2.4.0+cu121:
  Successfully uninstalled dgl-2.4.0+cu121
Found existing installation: torchdata 0.7.1
Uninstalling torchdata-0.7.1:
  Successfully uninstalled torchdata-0.7.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires torchdata==0.11.0, but you have torchdata 0.7.1 which is incompatible.


In [13]:
%%writefile /kaggle/working/data_loader.py
import torch
import dgl
from torch.utils.data import Dataset, DataLoader, random_split
from dgllife.utils import ScaffoldSplitter


class MoleculeDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


class MoleculeDatasetWithSmiles(Dataset):
    """Wraps (graph, label, smiles) tuples. Exposes .smiles for dgllife's ScaffoldSplitter."""
    def __init__(self, data):
        self.data   = data
        self.smiles = [item[2] for item in data]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        g, y, smiles = self.data[idx]
        return g, y


# ===========================================================================
# STANDARD COLLATE FN (For Supervised Fine-Tuning on Tox21)
# ===========================================================================
def collate_fn(batch):
    graphs, labels = zip(*batch)
    batched_graph  = dgl.batch(graphs)
    batched_labels = torch.stack(labels, dim=0).squeeze(1)
    return batched_graph, batched_labels


# ===========================================================================
# DENOISING COLLATE FN (For Self-Supervised Pretraining on ToxCast)
# ===========================================================================
def collate_fn_denoising(batch, noise_scale=0.05):
    """
    Strips the toxicity labels, injects Gaussian noise into 3D coordinates,
    and returns the noise tensor (N, 3) as the target for the model to predict.
    """
    graphs, _ = zip(*batch)
    batched_graph = dgl.batch(graphs)

    pos = batched_graph.ndata['pos']
    noise = torch.randn_like(pos) * noise_scale
    batched_graph.ndata['pos'] = pos + noise

    return batched_graph, noise


# ===========================================================================
#  TOX21 — SCAFFOLD SPLIT
# ===========================================================================
def get_tox21_scaffold_loaders(path="/kaggle/input/datasets/prajwalnayakat/3d-tox21-with-smiles-and-one-hot/tox21_3d_egnn_dataset_scaffold_onehot.pt", batch_size=32):
    raw     = torch.load(path, weights_only=False)
    dataset = MoleculeDatasetWithSmiles(raw)

    train_set, val_set, test_set = ScaffoldSplitter.train_val_test_split(
        dataset,
        frac_train=0.8,
        frac_val=0.1,
        frac_test=0.1
    )

    print(f"Tox21 Scaffold Split → Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    return train_loader, val_loader, test_loader


# ===========================================================================
#  TOXCAST — SCAFFOLD SPLIT (Pretraining source via Denoising)
# ===========================================================================
def get_toxcast_loaders(path="/kaggle/input/datasets/prajwalnayakat/3d-toxcast-with-one-hot/toxcast_3d_egnn_dataset_onehot.pt", batch_size=32):
    raw     = torch.load(path, weights_only=False)
    dataset = MoleculeDatasetWithSmiles(raw)

    # Replaced random_split with ScaffoldSplitter to force structural generalization
    train_set, val_set, test_set = ScaffoldSplitter.train_val_test_split(
        dataset,
        frac_train=0.8,
        frac_val=0.1,
        frac_test=0.1
    )

    print(f"ToxCast Scaffold Split → Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn_denoising)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn_denoising)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False, collate_fn=collate_fn_denoising)

    return train_loader, val_loader, test_loader


toxcast_train_loader, toxcast_val_loader, toxcast_test_loader = get_toxcast_loaders()
tox21_train_loader, tox21_val_loader, tox21_test_loader = get_tox21_scaffold_loaders()

Overwriting /kaggle/working/data_loader.py


In [15]:
%%writefile /kaggle/working/egnn_layer.py
import torch
import torch.nn as nn
import dgl
import dgl.function as fn


class EGNNLayer(nn.Module):
    """
    One layer of an Equivariant Graph Neural Network (EGNN).
    Based on: "E(n) Equivariant Graph Neural Networks" (Satorras et al., 2021)

    What makes EGNN different from GINEConv:
    - GINEConv: uses pre-computed distances as extra edge features. Coordinates
      are static — they never change during message passing.
    - EGNN: computes distances live from pos during every forward pass, AND
      updates the 3D coordinates of every node as part of the layer itself.
      This means the geometry evolves as information flows through the network,
      making it sensitive to the actual 3D shape of the molecule.

    Per-layer operations:
    1. For every edge: compute distance from current pos, run edge MLP
    2. For every node: aggregate neighbour messages, run node MLP → new hidden state
    3. For every node: compute a weighted sum of relative position vectors → update pos
    """

    def __init__(self, hidden_dim, edge_attr_dim=5):
        super().__init__()

        # Edge MLP: takes [h_i, h_j, distance, edge_attr] → message
        # hidden_dim * 2 for the two node states + 1 for distance + edge_attr_dim for bond type
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1 + edge_attr_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

        # Node MLP: takes [h_i, aggregated messages] → new h_i
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # Coordinate MLP: takes edge message → scalar weight for pos update
        # Output is a single scalar that scales the relative position vector (pos_i - pos_j)
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

    def edge_message(self, edges):
        """
        Runs on every edge simultaneously.
        Computes the distance between the two endpoint atoms from their
        current positions, then feeds everything into the edge MLP.
        """
        # Relative position vector and its scalar distance
        rel_pos  = edges.src['pos'] - edges.dst['pos']          # (E, 3)
        distance = torch.sqrt((rel_pos ** 2).sum(dim=-1, keepdim=True) + 1e-8)    # (E, 1)

        # Concatenate: source hidden state, dest hidden state, distance, bond type
        edge_input = torch.cat([
            edges.src['h'],           # (E, hidden_dim)
            edges.dst['h'],           # (E, hidden_dim)
            distance,                  # (E, 1)
            edges.data['edge_attr'],   # (E, edge_attr_dim)
        ], dim=-1)

        message    = self.edge_mlp(edge_input)     # (E, hidden_dim)
        coord_weight = self.coord_mlp(message)     # (E, 1) — scalar for pos update

        return {
            'message':      message,
            'coord_weight': coord_weight * rel_pos,  # (E, 3) — weighted relative vector
        }

    def node_update(self, nodes):
        """
        Runs on every node simultaneously.
        Aggregates incoming messages and updates the node's hidden state.
        """
        # 'agg_msg' is the sum of all incoming messages (set by dgl after edge_message)
        node_input = torch.cat([nodes.data['h'], nodes.data['agg_msg']], dim=-1)
        new_h      = self.node_mlp(node_input)
        return {'h': new_h}

    def forward(self, g, h, pos, edge_attr):
        with g.local_scope():
            g.ndata['h']         = h
            g.ndata['pos']       = pos
            g.edata['edge_attr'] = edge_attr

            # Step 1: compute messages and coordinate weights along every edge
            g.apply_edges(self.edge_message)

            # Step 2: aggregate messages into each node
            g.update_all(fn.copy_e('message', 'm'), fn.sum('m', 'agg_msg'))

            # Step 3: update node hidden states
            g.apply_nodes(self.node_update)

            # Step 4: update coordinates
            # Sum the weighted relative vectors arriving at each node
            g.update_all(fn.copy_e('coord_weight', 'cw'), fn.sum('cw', 'agg_cw'))
            new_pos = pos + g.ndata['agg_cw']   # (N, 3)

            return g.ndata['h'], new_pos


Overwriting /kaggle/working/egnn_layer.py


In [17]:
%%writefile /kaggle/working/train_ToxCast.py
import torch
import time
import torch.nn as nn
import torch.optim as optim
import dgl
from egnn_layer import EGNNLayer
from data_loader import toxcast_train_loader, toxcast_val_loader

torch.manual_seed(42)

# out_features is determined automatically from the sparsity-filtered column list
# saved during preprocessing — no manual number to keep in sync by hand.
VALID_COLUMNS_PATH = "/kaggle/input/datasets/prajwalnayakat/3d-toxcast-with-one-hot/toxcast_valid_columns.pt"  # ⚠️ update to match your Kaggle input
NUM_TASKS = torch.load(VALID_COLUMNS_PATH, weights_only=False).shape[0]
print(f"ToxCast pretraining task count (post sparsity filtering): {NUM_TASKS}")


# ==========================================
# 1. MODEL ARCHITECTURE
#    in_node_features=20 — one-hot FGN version (7 chemistry + is_fgn + 12 FG type)
#    No dropout in the prediction head here — this is the pretraining stage.
#    A dropout version gets swapped in during Tox21 fine-tuning, same pattern
#    as the QM9 pipeline: pretrain with a clean head, then replace it.
# ==========================================
class MoleColyteEGNN(nn.Module):
    def __init__(self, in_node_features=20, hidden_dim=128, edge_attr_dim=5, num_layers=3, out_features=3):
        super().__init__()

        self.input_proj = nn.Linear(in_node_features, hidden_dim)

        self.egnn_layers = nn.ModuleList([
            EGNNLayer(hidden_dim=hidden_dim, edge_attr_dim=edge_attr_dim)
            for _ in range(num_layers)
        ])

        self.prediction_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, out_features)  # Predicts 3D noise vector (dx, dy, dz) per atom
        )

    def forward(self, g, x, pos, edge_attr):
        h = self.input_proj(x)

        for layer in self.egnn_layers:
            h, pos = layer(g, h, pos, edge_attr)

        g.ndata['h'] = h

        # Return per-node predictions directly (N, 3) — no mean_nodes pooling!
        return self.prediction_head(h)



# ==========================================
# 2. PRETRAINING LOOP
# ==========================================
def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Kiln: {device}")
    print("Pretraining EGNN via 3D Coordinate Denoising on ToxCast")

    model     = MoleColyteEGNN(out_features=3).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    EPOCHS    = 50
    best_loss = float('inf')

    for epoch in range(EPOCHS):
        # ------------------------------------------
        # TRAINING PHASE
        # ------------------------------------------
        model.train()
        total_train_loss = 0
        print(f"Starting Epoch {epoch + 1}")

        for step, (batch_graph, noise_targets) in enumerate(toxcast_train_loader):
            batch_graph   = batch_graph.to(device)
            noise_targets = noise_targets.to(device)

            x         = batch_graph.ndata['x'].to(torch.float)
            pos       = batch_graph.ndata['pos'].to(torch.float)
            edge_attr = batch_graph.edata['edge_attr'].to(torch.float)

            optimizer.zero_grad()

            predictions = model(batch_graph, x, pos, edge_attr)   # (N_nodes, 3)
            loss        = criterion(predictions, noise_targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_train_loss += loss.item()

            if step % 100 == 0:
                print(f"\t Epoch {epoch + 1}/{EPOCHS} | Batch {step} | MSE Loss: {loss.item():.6f}")

        avg_train_loss = total_train_loss / len(toxcast_train_loader)
        print(f"==> Epoch {epoch + 1} Train Complete | Average MSE Loss: {avg_train_loss:.6f}")

        # ------------------------------------------
        # VALIDATION PHASE
        # ------------------------------------------
        model.eval()
        total_val_loss = 0

        with torch.no_grad():
            for batch_graph, noise_targets in toxcast_val_loader:
                batch_graph   = batch_graph.to(device)
                noise_targets = noise_targets.to(device)

                x         = batch_graph.ndata['x'].to(torch.float)
                pos       = batch_graph.ndata['pos'].to(torch.float)
                edge_attr = batch_graph.edata['edge_attr'].to(torch.float)

                predictions = model(batch_graph, x, pos, edge_attr)
                loss        = criterion(predictions, noise_targets)

                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(toxcast_val_loader)
        print(f"\tValidation Phase | Avg Val Loss: {avg_val_loss:.6f}")

        # ------------------------------------------
        # CHECKPOINTING
        # ------------------------------------------
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            torch.save(model.state_dict(), "/kaggle/working/egnn_toxcast_pretrained_best.pt")
            print(f"🏆 New best ToxCast denoising model saved! (Lowest Val Loss: {best_loss:.6f})\n")
        else:
            print(f"Model did not improve. Best Val loss remains: {best_loss:.6f}\n")

if __name__ == "__main__":
    since = time.time()
    train()
    print(f"Time: {(time.time() - since) / 60:.2f} minutes.")


Overwriting /kaggle/working/train_ToxCast.py


In [18]:
!pip install "numpy<2"

In [21]:
!python train_ToxCast.py

Start initializing RDKit molecule instances...
Creating RDKit molecule instance 1000/5625
Creating RDKit molecule instance 2000/5625
Creating RDKit molecule instance 3000/5625
Creating RDKit molecule instance 4000/5625
Creating RDKit molecule instance 5000/5625
Start computing Bemis-Murcko scaffolds.
Computing Bemis-Murcko for compound 1000/5625
Computing Bemis-Murcko for compound 2000/5625
Computing Bemis-Murcko for compound 3000/5625
Computing Bemis-Murcko for compound 4000/5625
Computing Bemis-Murcko for compound 5000/5625
ToxCast Scaffold Split → Train: 4500 | Val: 562 | Test: 563
Start initializing RDKit molecule instances...
Creating RDKit molecule instance 1000/5525
Creating RDKit molecule instance 2000/5525
Creating RDKit molecule instance 3000/5525
Creating RDKit molecule instance 4000/5525
Creating RDKit molecule instance 5000/5525
Start computing Bemis-Murcko scaffolds.
Computing Bemis-Murcko for compound 1000/5525
Computing Bemis-Murcko for compound 2000/5525
Computing Bemi